# 사전구축 라이브러리 — `langgraph-supervisor`

앞서 멀티에이전트 단원에서 Supervisor 구조를 **손으로** 짰다 (Router 구조화출력 + Command 라우팅). 같은 패턴을 라이브러리가 미리 만들어 제공한다 → **`langgraph-supervisor`** 의 `create_supervisor` 한 번이면 끝.

직접 구현과 비교:
- 직접: `make_supervisor_node` + Router 스키마 + 각 작업자 노드 + 그래프 조립
- prebuilt: `create_supervisor([agent들], model=, prompt=)` 한 줄

예제: 자료검색 전문가 + 코딩 전문가를 supervisor 가 지휘.

> `pip/uv install langgraph-supervisor`. `OPENAI_API_KEY`(+ 검색은 `TAVILY_API_KEY`) 필요.

## 환경 변수 준비

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 가 .env 에 없습니다"
print("환경변수 로드 완료")

## 1. 가장 단순한 예제 — 수학 전문가 + 리서치 전문가

[prebuilt] `create_react_agent` 로 전문가 에이전트를 만들 때 **`name`** 을 꼭 준다 — supervisor 가 이 이름으로 작업자를 호출한다. 그다음 `create_supervisor` 로 묶는다.

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph_supervisor import create_supervisor
from langgraph.prebuilt import create_react_agent

model = ChatOpenAI(model="gpt-4o")

def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b

def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b

def web_search(query: str) -> str:
    """Search the web for information."""
    return (
        "FAANG 2024 headcounts:\n"
        "Meta 67,317 / Apple 164,000 / Amazon 1,551,000 / Netflix 14,000 / Alphabet 181,269"
    )

math_agent = create_react_agent(
    model=model, tools=[add, multiply], name="math_expert",
    prompt="You are a math expert. Always use one tool at a time.",
)
research_agent = create_react_agent(
    model=model, tools=[web_search], name="research_expert",
    prompt="You are a world class researcher with access to web search. Do not do any math.",
)

In [ ]:
# create_supervisor: 작업자 리스트 + 관리 프롬프트만 주면 supervisor 그래프 완성
workflow = create_supervisor(
    [research_agent, math_agent],
    model=model,
    prompt=(
        "You are a team supervisor managing a research_expert and a math_expert. "
        "For current events, use research_expert. For math problems, use math_expert."
    ),
)
app = workflow.compile()

result = app.invoke({
    "messages": [{"role": "user",
        "content": "what's the combined headcount of the FAANG companies in 2024?"}]
})
for msg in result["messages"]:
    msg.pretty_print()

In [ ]:
from IPython.display import Image, display

try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    print(app.get_graph().draw_mermaid())

## 2. 자료검색 + 코딩 전문가 팀

이번엔 실제 도구로: 검색 전문가(Tavily) + 코딩 전문가(파이썬 실행). 구조는 동일, 작업자만 바뀐다.

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults
from typing import Annotated
from langchain_core.tools import tool

assert os.environ.get("TAVILY_API_KEY"), "TAVILY_API_KEY 필요"
llm = ChatOpenAI(model="gpt-4o", temperature=0)

search_agent = create_react_agent(
    llm, [TavilySearchResults(max_results=5)], name="research_expert",
    prompt="You are a world class researcher with web search. Do not do any coding.",
)

@tool
def python_exec_tool(code: Annotated[str, "python code to execute"]):
    """Execute python code. Print values you want to see with print(...)."""
    try:
        result = exec(code)
    except BaseException as e:
        return f"Failed to execute. Error: {repr(e)}"
    return f"Successfully executed:\n{code}\nStdout: {result}"

coding_agent = create_react_agent(
    llm, [python_exec_tool], name="coding_expert",
    prompt="You can only generate python code. Do not do any search.",
)

In [ ]:
graph = create_supervisor(
    [search_agent, coding_agent],
    model=llm,
    prompt=(
        "You are a team supervisor managing a research_expert and a coding_expert. "
        "For current events, use research_expert. For coding problems, use coding_expert."
    ),
).compile()

In [ ]:
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())

## 테스트
[multi_agent] `subgraphs=True` 로 전문가 에이전트 내부 진행까지 본다.

In [ ]:
for node, chunk in graph.stream(
    {"messages": [{"role": "user",
        "content": "첫째 항이 1인 피보나치 수열 함수를 파이썬으로 작성하고 10번째 항까지 출력해줘."}]},
    subgraphs=True, stream_mode="updates",
):
    if len(node) > 0:
        print("\n=====", node, "=====")
    for state_key, state_value in chunk.items():
        if isinstance(state_value, dict) and state_value.get("messages"):
            state_value["messages"][-1].pretty_print()

## 정리

- **`langgraph-supervisor`** = 직접 짜던 Supervisor 패턴의 prebuilt 버전
- `create_react_agent(..., name=...)` 로 전문가 만들고 → `create_supervisor([...], model, prompt)` 로 묶기
- 직접 구현(make_supervisor_node + Router + 노드 조립)을 한 줄로 대체 — 빠른 프로토타이핑에 유리
- 세밀한 제어가 필요하면 직접 구현, 표준 패턴이면 prebuilt

다음: 에이전트끼리 동적으로 제어권을 넘기는 **Swarm** prebuilt (`langgraph-swarm`).